# Protocolo ECDH (Elliptic Curve Diffie-Hellman): Intercambio de Claves Seguro

El protocolo **ECDH (Elliptic Curve Diffie-Hellman)** es un método criptográfico que permite a dos partes, tradicionalmente Alice y Bob, establecer una clave secreta compartida a través de un canal de comunicación inseguro. Esta clave puede usarse luego para cifrar su comunicación, aunque ECDH por sí solo no provee autenticación.

## 🚀 ¿Qué es ECDH y por qué usarlo?

ECDH es la versión de curva elíptica del clásico algoritmo Diffie-Hellman. Su principal ventaja es que logra el mismo nivel de seguridad con claves mucho más cortas, lo que se traduce en mayor eficiencia computacional y menor uso de ancho de banda, ideal para dispositivos con recursos limitados o redes con alta latencia.

### 1. Instalación de Dependencias
Necesitamos `fastecdsa` para las operaciones de curva elíptica y `secrets` para la generación segura de números aleatorios.

In [2]:
!pip install fastecdsa

### 2. Repaso Rápido: Criptografía de Curva Elíptica (ECC)

ECDH se basa en el **Problema del Logaritmo Discreto de Curva Elíptica (ECDLP)**. En una curva elíptica con un punto generador $G$, es fácil calcular $Q = d \cdot G$ (multiplicar un punto $G$ por un escalar $d$ para obtener un nuevo punto $Q$). Sin embargo, si solo conoces $Q$ y $G$, es extremadamente difícil encontrar $d$. Este $d$ es tu clave privada, y $Q$ es tu clave pública.

Usaremos la curva **secp256k1**, la misma que usa Bitcoin y Ethereum, para nuestro ejemplo.

In [3]:
from fastecdsa.curve import secp256k1
from fastecdsa.point import Point
import secrets

# Parámetros públicos de la curva (conocidos por todos)
G = secp256k1.G  # Punto generador
n = secp256k1.q  # Orden del grupo (rango de las claves privadas)

print(f"Punto Generador G: ({G.x}, {G.y})")
print(f"Orden de la curva n (rango de claves privadas): {hex(n)}")

Punto Generador G: (55066263022277343669578718895168534326250603453777594175500187360389116729240, 32670510020758816978083085130507043184471273380659243275938904335757337482424)
Orden de la curva n (rango de claves privadas): 0xfffffffffffffffffffffffffffffffebaaedce6af48a03bbfd25e8cd0364141


### 3. El Protocolo Alice y Bob (Paso a Paso)

#### Paso 1: Generación de Claves Privadas Secretas
Alice y Bob eligen cada uno un número entero aleatorio y secreto dentro del rango $[1, n-1]$. Estos serán sus claves privadas.

#### Paso 2: Cálculo de Claves Públicas Intermedias
Cada uno calcula un punto público en la curva multiplicando su clave privada por el punto generador $G$. Estos puntos se intercambian públicamente.

#### Paso 3: Derivación de la Clave Secreta Compartida
Cada uno utiliza la clave pública recibida del otro y su propia clave privada para calcular la misma clave secreta compartida.

### 3.1 Ejemplo Práctico: Intercambio de Claves ECDH entre Alice y Bob

A continuación, se muestra cómo Alice y Bob generan sus claves, las intercambian y derivan la misma clave secreta compartida, la cual solo ellos conocen. Este proceso es fundamental para establecer canales de comunicación seguros.

In [4]:
# === ALICE ===
# 1. Alice elige una clave privada (un número aleatorio secreto)
alice_private_key = secrets.randbelow(n)
print(f"Alice's Private Key (a): {hex(alice_private_key)}\n")

# 2. Alice calcula su clave pública (punto en la curva)
alice_public_key = alice_private_key * G
print(f"Alice's Public Key (A = a * G): ({hex(alice_public_key.x)}, {hex(alice_public_key.y)})\n")

# === BOB ===
# 1. Bob elige una clave privada (un número aleatorio secreto)
bob_private_key = secrets.randbelow(n)
print(f"Bob's Private Key (b): {hex(bob_private_key)}\n")

# 2. Bob calcula su clave pública (punto en la curva)
bob_public_key = bob_private_key * G
print(f"Bob's Public Key (B = b * G): ({hex(bob_public_key.x)}, {hex(bob_public_key.y)})\n")

print("--------------------------------------------------\n")
print("*** Alice y Bob se intercambian sus Claves Públicas (A y B) ***\n")
print("--------------------------------------------------\n")

# === ALICE (deriva la clave secreta compartida) ===
# Alice usa su clave privada (a) y la clave pública de Bob (B)
shared_secret_alice = alice_private_key * bob_public_key
print(f"Alice's Shared Secret (S = a * B): ({hex(shared_secret_alice.x)}, {hex(shared_secret_alice.y)})\n")

# === BOB (deriva la clave secreta compartida) ===
# Bob usa su clave privada (b) y la clave pública de Alice (A)
shared_secret_bob = bob_private_key * alice_public_key
print(f"Bob's Shared Secret (S = b * A): ({hex(shared_secret_bob.x)}, {hex(shared_secret_bob.y)})\n")

# === VERIFICACIÓN ===
print("--------------------------------------------------\n")
if shared_secret_alice == shared_secret_bob:
    print("✅ ¡Intercambio de clave exitoso! Alice y Bob tienen la misma clave secreta compartida.")
else:
    print("❌ Fallo en el intercambio de clave.")

print(f"\nClave Secreta Compartida (coordenada X): {hex(shared_secret_alice.x)}")

Alice's Private Key (a): 0x72f0d3cf208ca923f81057f1dbfe3e4c9d5baf7bbd7d9982ddf4d5ff068964e7

Alice's Public Key (A = a * G): (0x31a6eb62f1a5e2e9e6a08444f31a23415e8d67c858fcf93c23969aeaf8805531, 0xfaeecebffd02eb52b49adbb786f2a94188dff449043030b61beb6edd42e4c9b2)

Bob's Private Key (b): 0x2fb0b3c15fe9bc5953e6ccff59333ab6bed5c3f5fcb58a3cea305f0950751d1d

Bob's Public Key (B = b * G): (0x15a54e692e5df6bf1915de858a40d238a1a69176b9708879b56673fc4410d3b8, 0x1b8dc2dfde82279d6b5984849e443056b5ec11c9fc7daa26fb08d6e5a86922a4)

--------------------------------------------------

*** Alice y Bob se intercambian sus Claves Públicas (A y B) ***

--------------------------------------------------

Alice's Shared Secret (S = a * B): (0xc94fcdfcdf8cfa68c24bb2d70cbfdc8265cdf6434f11c13af622b9685b6fd882, 0xe7e288871c733a5a45ba337d390ba717af3de48a8a80f2ab27356279efff044b)

Bob's Shared Secret (S = b * A): (0xc94fcdfcdf8cfa68c24bb2d70cbfdc8265cdf6434f11c13af622b9685b6fd882, 0xe7e288871c733a5a45ba337d390ba717

## 🔐 Fortalezas de ECDH

1.  **Perfect Forward Secrecy (PFS)**: Si las claves de largo plazo de Alice o Bob se ven comprometidas en el futuro, las claves de sesión generadas con ECDH en el pasado (y ya usadas) no pueden ser descifradas. Esto se debe a que la clave compartida efímera nunca se transmite directamente ni se almacena a largo plazo.
2.  **Eficiencia**: Como todas las criptografías de curva elíptica, ECDH ofrece la misma seguridad que métodos RSA/Diffie-Hellman con claves de mayor longitud, pero con un tamaño de clave mucho menor, lo que reduce la carga computacional.

## ⚠️ Falencias y Vulnerabilidades de ECDH (Sin Autenticación)

La principal debilidad de ECDH (y Diffie-Hellman en general) es su **ausencia de autenticación intrínseca**. Esto significa que Alice y Bob no tienen forma de saber si la clave pública que recibieron realmente proviene de la persona con la que creen que están hablando.

### Ataque Man-in-the-Middle (MITM)
Un atacante (Eve) puede explotar esta falencia para interceptar y manipular la comunicación:

1.  **Eve se intercepta a sí misma**: Cuando Alice envía su clave pública $A$ a Bob, Eve la intercepta y envía a Bob su propia clave pública $E_1$.
2.  **Bob recibe de Eve**: Bob cree que recibió $A$ de Alice, pero en realidad es $E_1$ de Eve. Bob entonces calcula una clave secreta compartida con Eve: $S_{Bob-Eve} = b \cdot E_1$.
3.  **Eve se hace pasar por Bob**: Cuando Bob envía su clave pública $B$ a Alice, Eve la intercepta y envía a Alice su propia clave pública $E_2$.
4.  **Alice recibe de Eve**: Alice cree que recibió $B$ de Bob, pero en realidad es $E_2$ de Eve. Alice entonces calcula una clave secreta compartida con Eve: $S_{Alice-Eve} = a \cdot E_2$.

**Resultado**: Eve ahora comparte una clave secreta con Alice ($S_{Alice-Eve}$) y otra clave secreta con Bob ($S_{Bob-Eve}$). Eve puede descifrar el tráfico de Alice, re-cifrarlo con su clave compartida con Bob, y enviarlo a Bob. De esta manera, Alice y Bob creen que se están comunicando de forma segura entre sí, sin saber que Eve está leyendo y manipulando todo.

### Otras Consideraciones y Vulnerabilidades:

1.  **Generación de Números Aleatorios (Claves Privadas)**: Si el generador de números aleatorios utilizado para crear las claves privadas ($a$ y $b$) es predecible o tiene baja entropía, un atacante podría adivinar las claves privadas y, por lo tanto, las claves compartidas.
2.  **Parámetros de la Curva**: La seguridad depende de la correcta elección de la curva elíptica y su punto generador. Curvas mal elegidas o débiles podrían ser vulnerables a ataques conocidos.
3.  **Implementación**: Errores en la implementación del algoritmo pueden introducir vulnerabilidades (ej. fallos de temporización, manejo incorrecto de puntos en el infinito).
4.  **Ataques de Small Subgroup**: Algunas implementaciones históricas tenían vulnerabilidades si los puntos públicos compartidos pertenecían a un subgrupo pequeño de la curva.

### Solución: Autenticación con Firmas Digitales (ECDSA)

Para mitigar el ataque MITM, ECDH se combina comúnmente con algoritmos de **firma digital** como **ECDSA (Elliptic Curve Digital Signature Algorithm)**. 

Antes de intercambiar claves públicas en ECDH, Alice y Bob se autentican mutuamente firmando sus claves públicas ECDH con sus propias claves de firma de largo plazo. Esto asegura que cada uno sabe que la clave pública que recibe proviene del interlocutor esperado y no de un atacante. Este es el principio detrás de protocolos como TLS/SSL.